# Trainer implementation test

In [ ]:
import jax 
import diffrax as dfx
import distrax as dsx
import equinox as eqx
import optax
import jax
import jax.numpy as jnp

from superiorflows import Flow

class MLPVelocity(eqx.Module):
    """MLP velocity field with time conditioning."""

    mlp: eqx.nn.MLP

    def __init__(self, input_dim: int, width: int, depth: int, *, key):
        self.mlp = eqx.nn.MLP(
            in_size=input_dim + 1,
            out_size=input_dim,
            width_size=width,
            depth=depth,
            activation=jax.nn.tanh,
            key=key,
        )

    def __call__(self, t, x, args):
        t_feat = jnp.broadcast_to(t, x.shape[:-1] + (1,))
        return self.mlp(jnp.concatenate([x, t_feat], axis=-1))

def make_ml_loss(base_distrbution, **kwargs):

    @eqx.filter_jit
    def maximum_likelihood_loss(model, batch, key=None):
        flow = Flow(
            velocity_field=model,
            base_distribution=base_distrbution,
            **kwargs,
            )
        return -jnp.mean(flow.log_prob(batch))

    return maximum_likelihood_loss
    

@eqx.filter_jit
def training_step(model, opt_state, batch, key, loss_function): 
    loss, grads = eqx.filter_value_and_grad(loss_function)(model, batch, key)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss


key = jax.random.key(0)
d = 2
key, subkey = jax.random.split(key)
model = MLPVelocity(input_dim=d, width=16, depth=3, key=subkey)
base_dist = dsx.MultivariateNormalDiag(jnp.zeros(d), jnp.ones(d))

key, subkey = jax.random.split(key)
batch = base_dist.sample(seed=subkey, sample_shape=(10,)) # Using base as samples for testing, should be target samples
loss_function = make_ml_loss(base_dist, stepsize_controller=dfx.PIDController(rtol=1e-6, atol=1e-6))

loss_function(model, batch)

Array(2.7974286, dtype=float32)

In [25]:
loss, grads = eqx.filter_value_and_grad(loss_function)(model, batch)

In [ ]:
def train(model, opt_state,data_loader, epochs, key, loss_function):
    for epoch in range(epochs):
        for batch in data_loader:
            key, subkey = jax.random.split(key)
            model, opt_state, loss = training_step(model, opt_state, batch, subkey, loss_function)
    return model, opt_state

In [ ]:
import grain


<function iter>